In [3]:
from __future__ import annotations
from pathlib import Path
import json
import re
from nltk.tokenize import TweetTokenizer 
from collections import Counter
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')
nltk.download('punkt')
import pandas as pd
from sklearn.model_selection import train_test_split

# SENTIMENT ANALYSIS
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

nltk.download("vader_lexicon")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [4]:
#set up
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

TWEETS_PATH = BASE_DIR / "tweets.txt"
EMOJIS_PATH = BASE_DIR / "emoji.txt"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.15
VALID_SIZE = 0.15

In [5]:
TWEETS_PATH = Path("tweets.txt")
EMOJIS_PATH = Path("emoji.txt")

def load_data() -> pd.DataFrame:
    tweets = open(TWEETS_PATH, encoding="utf-8").read().splitlines()
    emojis = open(EMOJIS_PATH, encoding="utf-8").read().splitlines()
    tweets_emojis_df = pd.DataFrame({
        "text": tweets,
        "label": emojis,
    })
    return tweets_emojis_df

load_data()

,text,label
0,RT @VibingOverHoes: Bet you'll get hungry htt...,heart_eyes
1,Starbucks employee confuses boyfriend by sayin...,yum
2,When your Starbucks store makes you an iced mo...,sob
3,"Being told ""girl your romper looks fierce!"" At...",blush
4,"I got a Starbucks drink at school today, shit ...",sob
...,...,...
225326,RT @TheFunnyVine: She broke down as soon as sh...,grin
225327,@PinKC0ttNkandi_ Mines shorter than yours Lls ...,sob
225328,@BenColeyGolf was yours the 5k bet on US earli...,wink
225329,@Kevyn_Brown @BurgerKing come on leek i gotta ...,sob


In [6]:
# Clean the tweets text so that unnecessary information is removed
# Added another column to the df with the cleaned text
def clean_text(text):
    text = re.sub(r'^RT[\s]+', '', text)       # remove RT
    text = text.lower()
    text = re.sub(r'http\S+', '', text)        # remove links
    text = re.sub(r'@\w+', '', text)           # remove mentions
    text = re.sub(r'#', '', text)              # remove hashtag symbol
    text = re.sub(r'(.)\1{2,}', r'\1\1', text) # reduce repeated letters
    text = re.sub(r'[^\w\s\']', '', text)  # remove punctuation except '
    text = re.sub(r'\s+', ' ', text).strip()   # remove extra spaces
    return text


In [7]:
#extra simple features for analysis

# Sentiment Analysis:
# Sentiment scores ranged from 0 to 1 for negative, neutral, and positive components, 
# with the compound score ranging from -1 to 1. 
# Tweets were categorized as positive, negative, or neutral using 
# standard VADER thresholds (≥ 0.05 for positive, ≤ -0.05 for negative).
# sent_neg = proportion of negative words
# sent_neu = proportion of neutral words
# sent_pos = proportion of positive words
# sent_compound = all sentiment into a single score


sia = SentimentIntensityAnalyzer()

BRANDS = [
    "starbucks",
    "mcdonalds",
    "subway",
    "dominos",
    "dunkindonuts",
    "pizzahut",
    "walmart",
    "burger king",
    "chipotle",
    "five guys"
]

def add_simple_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["clean_text"] = df["text"].apply(clean_text)
    df["char_count"] = df["text"].str.len()
    df["word_count"] = df["clean_text"].str.split().str.len()
    df["has_link"] = df["text"].str.contains(r"http\S+", regex=True)
    df["has_mention"] = df["text"].str.contains(r"@\w+", regex=True)
    df["has_hashtag"] = df["text"].str.contains(r"#\w+", regex=True)
    df["is_retweet"] = df["text"].str.contains(r"^RT[\s]+", regex=True)
    df["exclamation_count"] = df["text"].str.count(r"!")
    df["question_count"] = df["text"].str.count(r"\?")

   
    # SENTIMENT FEATURES (NEW)
    
    sentiment_scores = df["clean_text"].apply(lambda x: sia.polarity_scores(x))

    df["sent_neg"] = sentiment_scores.apply(lambda x: x["neg"])
    df["sent_neu"] = sentiment_scores.apply(lambda x: x["neu"])
    df["sent_pos"] = sentiment_scores.apply(lambda x: x["pos"])
    df["sent_compound"] = sentiment_scores.apply(lambda x: x["compound"])

    # BRAND FEATURES (NEW)

    text_lower = df["text"].str.lower()

    brand_cols = []
    for brand in BRANDS:
        col = "brand_" + brand.replace(" ", "_")
        brand_cols.append(col)

        # handle multi-word brands vs single-word brands
        if " " in brand:
            pattern = re.escape(brand)
        else:
            pattern = r"\b" + re.escape(brand) + r"\b"

        df[col] = text_lower.str.contains(pattern, regex=True, na=False).astype(int)

    # total number of brand mentions in each tweet
    df["brand_mention_total"] = df[brand_cols].sum(axis=1)

    return df


In [10]:
tweets_emojis_df = load_data()
tweets_emojis_df = add_simple_features(tweets_emojis_df)
print(tweets_emojis_df.head())
tweets_emojis_df[
    [
        "text",
        "clean_text",
        "char_count",
        "word_count",
        "has_link",
        "has_mention",
        "has_hashtag",
        "is_retweet",
        "exclamation_count",
        "question_count",
        
        # SENTIMENT COLS (NEW)
        "sent_neg",
        "sent_neu",
        "sent_pos",
        "sent_compound",

        "brand_starbucks",
        "brand_mcdonalds",
        "brand_chipotle",
        "brand_mention_total"
    ]
].head(10)

                                                text       label  \
0  RT @VibingOverHoes: Bet you'll get hungry  htt...  heart_eyes   
1  Starbucks employee confuses boyfriend by sayin...         yum   
2  When your Starbucks store makes you an iced mo...         sob   
3  Being told "girl your romper looks fierce!" At...       blush   
4  I got a Starbucks drink at school today, shit ...         sob   

                                          clean_text  char_count  word_count  \
0                              bet you'll get hungry          66           4   
1  starbucks employee confuses boyfriend by sayin...          65          10   
2  when your starbucks store makes you an iced mo...          73          13   
3  being told girl your romper looks fierce at th...         135          25   
4  i got a starbucks drink at school today shit t...          97          19   

   has_link  has_mention  has_hashtag  is_retweet  exclamation_count  ...  \
0      True         True        F

,text,clean_text,char_count,word_count,has_link,has_mention,has_hashtag,is_retweet,exclamation_count,question_count,sent_neg,sent_neu,sent_pos,sent_compound,brand_starbucks,brand_mcdonalds,brand_chipotle,brand_mention_total
0,RT @VibingOverHoes: Bet you'll get hungry htt...,bet you'll get hungry,66,4,True,True,False,True,0,0,0.000,1.000,0.000,0.0000,0,0,0,0
1,Starbucks employee confuses boyfriend by sayin...,starbucks employee confuses boyfriend by sayin...,65,10,False,False,False,False,0,0,0.204,0.796,0.000,-0.3182,1,0,0,1
2,When your Starbucks store makes you an iced mo...,when your starbucks store makes you an iced mo...,73,13,False,False,False,False,0,0,0.000,1.000,0.000,0.0000,1,0,0,1
3,"Being told ""girl your romper looks fierce!"" At...",being told girl your romper looks fierce at th...,135,25,False,False,False,False,1,0,0.000,0.835,0.165,0.6948,0,0,0,1
4,"I got a Starbucks drink at school today, shit ...",i got a starbucks drink at school today shit t...,97,19,False,False,False,False,0,0,0.454,0.546,0.000,-0.8957,1,0,0,1
5,So when is Starbucks getting their grill chees...,so when is starbucks getting their grill chees...,55,9,False,False,False,False,0,1,0.000,1.000,0.000,0.0000,1,0,0,1
6,really need @Jade_morgann to bring me some su...,really need to bring me some subway,54,7,False,True,False,False,0,0,0.000,1.000,0.000,0.0000,0,0,0,1
7,Bet you're feeling super salty now. https://t....,bet you're feeling super salty now,59,6,True,False,False,False,0,0,0.000,0.426,0.574,0.6597,0,0,0,0
8,RT @Daniarmstrong88: So it's the end of #brang...,so it's the end of brangelina i bet a certain ...,121,18,False,True,True,True,0,0,0.090,0.621,0.288,0.3626,0,0,0,0
9,@ooohkarluuuh and forgot to get me Starbucks,and forgot to get me starbucks,45,6,False,True,False,False,0,0,0.000,1.000,0.000,0.0000,1,0,0,1


In [11]:
# Checking for Missing Data -> No missing data!
tweets_emojis_df.isnull().sum()

text                   0
label                  0
clean_text             0
char_count             0
word_count             0
has_link               0
has_mention            0
has_hashtag            0
is_retweet             0
exclamation_count      0
question_count         0
sent_neg               0
sent_neu               0
sent_pos               0
sent_compound          0
brand_starbucks        0
brand_mcdonalds        0
brand_subway           0
brand_dominos          0
brand_dunkindonuts     0
brand_pizzahut         0
brand_walmart          0
brand_burger_king      0
brand_chipotle         0
brand_five_guys        0
brand_mention_total    0
dtype: int64

In [12]:
def save_outputs(df: pd.DataFrame) -> None:
    train_valid_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=df["label"],
    )

    valid_relative = VALID_SIZE / (1 - TEST_SIZE)
    train_df, valid_df = train_test_split(
        train_valid_df,
        test_size=valid_relative,
        random_state=RANDOM_STATE,
        stratify=train_valid_df["label"],
    )

    df.to_csv(OUTPUT_DIR / "tweets_emojis_clean.csv", index=False)
    train_df.to_csv(OUTPUT_DIR / "train.csv", index=False)
    valid_df.to_csv(OUTPUT_DIR / "valid.csv", index=False)
    test_df.to_csv(OUTPUT_DIR / "test.csv", index=False)

    label_counts = df["label"].value_counts().reset_index()
    label_counts.columns = ["label", "count"]
    label_counts.to_csv(OUTPUT_DIR / "label_counts.csv", index=False)

    dataset_summary = {
        "n_rows": int(len(df)),
        "n_classes": int(df["label"].nunique()),
        "avg_char_count": float(df["char_count"].mean()),
        "avg_word_count": float(df["word_count"].mean()),
        "pct_with_link": float(df["has_link"].mean()),
        "pct_with_mention": float(df["has_mention"].mean()),
        "pct_with_hashtag": float(df["has_hashtag"].mean()),
        "pct_retweets": float(df["is_retweet"].mean()),

        # include sentiment cols
        "avg_sent_neg": float(df["sent_neg"].mean()),
        "avg_sent_neu": float(df["sent_neu"].mean()),
        "avg_sent_pos": float(df["sent_pos"].mean()),
        "avg_sent_compound": float(df["sent_compound"].mean()),
        "min_sent_compound": float(df["sent_compound"].min()),
        "max_sent_compound": float(df["sent_compound"].max()),
    }

    with open(OUTPUT_DIR / "dataset_summary.json", "w", encoding="utf-8") as f:
        json.dump(dataset_summary, f, indent=2)

    split_summary = pd.DataFrame([
        {"split": "train", "rows": len(train_df)},
        {"split": "valid", "rows": len(valid_df)},
        {"split": "test", "rows": len(test_df)},
    ])
    split_summary.to_csv(OUTPUT_DIR / "split_summary.csv", index=False)

    print(f"Saved cleaned dataset to: {OUTPUT_DIR / 'tweets_emojis_clean.csv'}")
    print(f"Saved train/valid/test files in: {OUTPUT_DIR}")

save_outputs(tweets_emojis_df)


Saved cleaned dataset to: /datasets/_deepnote_work/outputs/tweets_emojis_clean.csv
Saved train/valid/test files in: /datasets/_deepnote_work/outputs


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=62b91583-1382-466d-9446-3dc5e725c312' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>